In [ ]:
from typing import Literal
import time
from enum import Enum
import os

FLOW = Literal["DIRECT", "PIVOT_LLM", "PIVOT_RB", "PIVOT_RB_LLM"]

PIPELINE_TYPE: FLOW = "DIRECT"

# MODEL_ID = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"
# MODEL_ID = "bigcode/starcoder2-15b"
# MODEL_ID = "ise-uiuc/Magicoder-S-DS-6.7B"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_ID = "microsoft/wavecoder-ultra-6.7b"

HF_TOKEN = os.getenv("HF_TOKEN")

### ----- Directory ----- ###

PROJECT_DIR = "drive/MyDrive/Colab Notebooks/LoResLift/HumanEval/"

class OutputPath(str, Enum):
    DIRECT = "results-direct"
    PIVOT_LLM = "results-pivot-llm"
    PIVOT_RB = "results-pivot-rb"
    PIVOT_RB_LLM = "results-pivot-rb-llm"

MODEL_NICKNAME = MODEL_ID.replace("/", "__")

OUTPUT_GROUP_DIR = os.path.join(PROJECT_DIR, "results", MODEL_NICKNAME)
OUTPUT_DIR = os.path.join(OUTPUT_GROUP_DIR, OutputPath[PIPELINE_TYPE])
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "output.jsonl")
OUTPUT_EVAL_FILE = os.path.join(OUTPUT_DIR, "output_eval.jsonl")
RESULTS_FILE = os.path.join(OUTPUT_DIR, "output_eval.jsonl_results.jsonl")

OUTPUT_PIVOT_DIR = os.path.join(OUTPUT_GROUP_DIR, "results-pivot")
OUTPUT_JAVA_FILE = os.path.join(OUTPUT_PIVOT_DIR, "java_output.jsonl")

JAVA_PROMPTS_DIR = os.path.join(PROJECT_DIR, "Prompts", "translated")

print(f"Starting pipeline flow: {PIPELINE_TYPE}")
notebook_start = time.time()

# --- Mount drive and create dirs ---

from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_PIVOT_DIR, exist_ok=True)


Starting pipeline flow: DIRECT
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Presets

## Env setup

In [ ]:
!pip install transformers evaluate accelerate huggingface_hub jsonlines
!pip install git+https://github.com/JetBrains-Research/mxeval.git
!pip install -U datasets

!rm -rf kotlin-compiler-1.9.24.zip
!wget https://github.com/JetBrains/kotlin/releases/download/v1.9.24/kotlin-compiler-1.9.24.zip
!unzip -o kotlin-compiler-1.9.24.zip
os.environ["PATH"] += ":/content/kotlinc/bin"

  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)
  Cloning https://github.com/JetBrains-Research/mxeval.git to /tmp/pip-req-build-47dfgwce
  Running command git clone --filter=blob:none --quiet https://github.com/JetBrains-Research/mxeval.git /tmp/pip-req-build-47dfgwce
  Resolved https://github.com/JetBrains-Research/mxeval.git to commit a29e7e4e79548a2f5e0b0b4c7e0eeb4de63c3681
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached datasets-4.6.0-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.6.0-py3-none-any.whl (520 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
--2026-02-27 15:53:20--  https://github.com/JetBrains/kotlin/releases/download/v1.9.24/kotli

## Load dataset

In [ ]:
import json

jsonl_path = "data.jsonl"  # path to your JSONL file

# Load JSONL
with open(jsonl_path, "r", encoding="utf-8") as f:
    problem_list = [json.loads(line) for line in f]

# Same structures as before
problem_dict = {item["task_id"]: item for item in problem_list}

kotlin_prompt_dict = {
    task_id: item["prompt"]
    for task_id, item in problem_dict.items()
}


# def build_java_prompt_dict():
#     prompt_dict = {}
#     target_files = ".java"

#     for filename in os.listdir(JAVA_PROMPTS_DIR):
#         if filename.endswith(target_files):
#             task_id = os.path.splitext(filename)[0]  # Remove '.kt'
#             full_task_id = f"HumanEval_kotlin/{task_id}"
#             filepath = os.path.join(JAVA_PROMPTS_DIR, filename)

#             with open(filepath, 'r', encoding='utf-8') as f:
#                 content = f.read()

#             prompt_dict[full_task_id] = content

#     return prompt_dict


# java_prompt_dict = build_java_prompt_dict()

# java_prompt_dict = {
#     task_id: content
#     for task_id, content in java_prompt_dict.items()
#     if task_id in problem_dict
# }


kotlin_signatures = {}
for key, item in problem_dict.items():
    lines = item["prompt"].strip().split('\n')
    kotlin_signatures[key] = lines[-1]
    problem_dict[key]["prompt"] = '\n'.join(lines[:-1])

print(kotlin_signatures)
print(problem_dict)

{'HumanEval_kotlin/0': 'fun hasCloseElements(numbers : List<Double>, threshold : Double) : Boolean {', 'HumanEval_kotlin/1': 'fun separateParenGroups(parenString : String) : List<String> {', 'HumanEval_kotlin/2': 'fun truncateNumber(number : Double) : Double {', 'HumanEval_kotlin/3': 'fun belowZero(operations : List<Any>) : Boolean {', 'HumanEval_kotlin/4': 'fun meanAbsoluteDeviation(numbers : List<Double>) : Double {', 'HumanEval_kotlin/5': 'fun intersperse(numbers : List<Any>, delimeter : Int) : List<Any> {', 'HumanEval_kotlin/6': 'fun parseNestedParens(parenString : String) : List<Int> {', 'HumanEval_kotlin/7': 'fun filterBySubstring(strings : List<Any>, substring : String) : List<Any> {', 'HumanEval_kotlin/8': 'fun sumProduct(numbers : List<Any>) : List<Int> {', 'HumanEval_kotlin/9': 'fun rollingMax(numbers : List<Any>) : List<Any> {', 'HumanEval_kotlin/10': 'fun makePalindrome(string : String) : String {', 'HumanEval_kotlin/11': 'fun stringXor(a : String, b : String) : String {', 

## Load LLM Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = MODEL_ID
token = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    token=token,
    trust_remote_code=True
)


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

## Commons, Utils

In [ ]:
import re
import torch
from transformers import GenerationConfig, StoppingCriteria, StoppingCriteriaList

class StopOnSecondFence(StoppingCriteria):
    def __init__(self, tokenizer, start_len: int, max_check_tokens: int = 2048):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.max_check_tokens = max_check_tokens

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        # Only inspect the generated tail (after the prompt)
        tail_ids = input_ids[0, self.start_len:].tolist()
        if not tail_ids:
            return False
        # Decode a bounded tail for efficiency
        tail_ids = tail_ids[-self.max_check_tokens:]
        text = self.tokenizer.decode(tail_ids, skip_special_tokens=True)
        return text.count("```") >= 2

class StoppingCriteriaSub(StoppingCriteria):
    def __init__(self, stops, tokenizer):
        (StoppingCriteria.__init__(self),)
        self.stops = rf"{stops}"
        self.tokenizer = tokenizer

    def __call__(
        self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs
    ) -> bool:
        last_three_tokens = [int(x) for x in input_ids.data[0][-3:]]
        decoded_last_three_tokens = self.tokenizer.decode(last_three_tokens)

        return bool(re.search(self.stops, decoded_last_three_tokens))

def generate(problem: str, model, tokenizer, device) -> str:
    criterion = StoppingCriteriaSub(stops="\n}\n", tokenizer=tokenizer)
    stopping_criteria = StoppingCriteriaList([criterion])

    inputs = tokenizer.encode(problem, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs,
        max_new_tokens=2048,
        min_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,      # Deterministic (ICSE-friendly)
        temperature=0.0,      # Ignored when do_sample=False
        top_p=1.0,
        num_beams=1,
        stopping_criteria=stopping_criteria,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def clean_answer(code: str, lang: str) -> str:
    # Remove block and line comments
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"//.*", "", code)

    lines = code.strip().split("\n")

    # Find the first line
    function_prefix = "fun " if lang == "kotlin" else "public "
    start_idx = -1
    for i, line in enumerate(lines):
        if line.strip().startswith(function_prefix):
            start_idx = i
            break

    if start_idx == -1:
        return code

    # Find the first line that starts with '}' (no indentation)
    end_idx = len(lines)
    for j in range(start_idx, len(lines)):
        if lines[j].startswith("}"):  # exactly no indentation
            end_idx = j + 1  # include the '}' line
            break

    return "\n".join(lines[start_idx:end_idx])


In [ ]:
import argparse
import json
import os
import re
import sys
import textwrap
from typing import Dict, Iterable, Tuple, Optional, List, Callable, Any

# --------------------------------
# Fenced block extraction (strict)
# --------------------------------
FENCE_RE = re.compile(r"(?s)```([A-Za-z0-9#+\-.]*)\s*\n(.*?)\n```")

def extract_single_fenced_code_block(text: str, expected_language: Optional[str] = None) -> Tuple[Optional[str], str]:
    """
    Return (language, code). Raises if none or >1 blocks found after filtering by expected_language.
    """
    matches = FENCE_RE.findall(text)
    if expected_language:
        matches = [(lang, body) for (lang, body) in matches if lang.lower() == expected_language.lower()]
    if len(matches) != 1:
        print(f"Found {len(matches)} fenced code blocks; expected exactly 1.")
        raise ValueError(f"Found {len(matches)} fenced code blocks; expected exactly 1.")
    lang, body = matches[0]
    return (lang or None), body.strip()

# -----------------------
# I/O helpers for JSONL
# -----------------------
def read_jsonl(path: str) -> Iterable[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                raise ValueError(f"Failed to parse JSON on line {line_no} of {path}: {e}")

def write_jsonl(records: Iterable[Dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def build_prompt(tokenizer, messages: list[dict[str, str]]) -> str:
    """
    Build the input string for the model.
    - If the tokenizer has a chat_template, use it.
    - Otherwise, fall back to concatenating message contents.
    """
    if getattr(tokenizer, "chat_template", None):
        # Chat-style model (e.g. llama-2-chat, mistral-instruct)
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        # Base/code model (e.g. StarCoder2)
        return "\n".join(m["content"] for m in messages)


# -----------------------
# LLM call (deterministic)
# -----------------------
def generate_once(tokenizer, model, messages: list[dict[str, str]], max_new_tokens: int = 3072) -> str:
    # 🔹 Only this block changes
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = "\n".join(m["content"] for m in messages)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start_len = inputs["input_ids"].shape[1]
    stops = StoppingCriteriaList([StopOnSecondFence(tokenizer, start_len)])

    gen_cfg = GenerationConfig(
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        eos_token_id=[tokenizer.eos_token_id],
    )

    outputs = model.generate(
        **inputs,
        generation_config=gen_cfg,
        stopping_criteria=stops,
    )
    gen_only = outputs[0, start_len:]
    return tokenizer.decode(gen_only, skip_special_tokens=True).strip()


def make_messages(**params):
    return [{"role": "user", "content": PROMPT_TEMPLATE.format(**params)}]



def run_generic_pipeline(
    out_dir: str,
    relevant_kotlin_samples: List[Any],
    get_extra_data: Optional[Callable[[str], Dict[str, Any]]],
    get_output_file_name: Callable[[str], str],
    out_jsonl: Optional[str],
):
    # Setup directories
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    results = []
    error_count = 0
    success_count = 0
    for idx, kotlin_sample in enumerate(relevant_kotlin_samples, 1):
        sid = kotlin_sample.get("task_id")
        prompt = kotlin_sample.get("prompt")
        kotlin_fn_signature = prompt.strip().split("\n")[-1]
        current_data = {
            "fn_description": kotlin_sample.get("description", "").strip(),
        }

        if get_extra_data is not None:
            extra_data = get_extra_data(sid)
            current_data.update(extra_data)

        # Build messages and generate
        messages = make_messages(**current_data)
        print(messages[0]["content"])
        raw = generate_once(tokenizer, model, messages)
        print(raw)

        # Extract code
        status = "ok"
        lang = None
        code = ""
        error = None
        try:
            lang, code = extract_single_fenced_code_block(raw, expected_language="kotlin")
        except Exception as e:
            status = "error"
            print(str(e))
            error_count += 1

        # Write files
        if status == "ok":
            fname = get_output_file_name(sid)
            path = os.path.join(out_dir, fname)
            with open(path, "w", encoding="utf-8") as f:
                f.write(code)

            results.append({
                "task_id": sid,
                "completion": code,
                "language": "kotlin",
            })
            success_count += 1

        print(f"[{idx}/{len(relevant_kotlin_samples)}] id={sid} status={status}")

    print(f"Errors: {error_count}/{len(relevant_kotlin_samples)}")
    print(f"Success: {success_count}/{len(relevant_kotlin_samples)}")

    if out_jsonl:
        write_jsonl(results, out_jsonl)
        print(f"Done. Wrote results to: {out_jsonl}")



## Prompts

In [ ]:
J2K_TRANSLATION_TEMPLATE = """You are an idiomatic code translator with expert-level knowledge of Kotlin and Java. Your task is to translate a Java function to Kotlin, using the Kotlin function signature provided.

You should aim to generate idiomatic Kotlin code, while preserving the semantics of the original Java code as much as possible. Try to maintain type safety. Replace unavailable APIs with the Kotlin standard library or the closest equivalent where possible.

Explain your reasoning concisely. You should consider inputs, outputs, control flow (conditions and loops), interactions with other functions, libraries and APIs.

Example 1:
Code:
```java
public static String numberToWord(int x) {{
    String msg;
    switch (x) {{
        case 1:
            msg = "one";
            break;
        case 2:
            msg = "two";
            break;
        default:
            msg = "other";
            break;
    }}
    return msg;
}}
```
Reasoning:
---
The "numberToWord" function takes as input an integer "x". It outputs a string that's either "one", "two", or "other". The "switch" expression branches on x: 1 -> "one", x: 2 -> "two", else -> "other".
---

Example 2:
Code:
```java
public static void printRange(int n) {{
    // inclusive (0..n)
    for (int i = 0; i <= n; i++) {{
        System.out.println(i);
    }}
    // exclusive (0 until n)
    for (int i = 0; i < n; i++) {{
        System.out.println(i);
    }}
}}
```
Reasoning:
---
The "printRange" function takes as input an integer "n". It operates two sequential loops which prints from 0 to n, the first loop using "<=" is inclusive, the second loop using "<" is exclusive.
---

Example 3:
Code:
```java
public static int max(int a, int b) {{
    if (isLarger(a, b)) {{
        return a;
    }} else {{
        return b;
    }}
}}
```
Reasoning:
---
The "max" function takes as input two integers "a" and "b" and returns the larger integer. It has an "if/else" expression to choose which number will be returned. The "isLarger" function is invoked by the "max" function to determine if integer "a" is larger than integer "b".
---

Use reasoning like this to assist your translation.

Output the reasoning in the block wrapped by '---'. Output the requested function and relevant imports in a fenced code block wrapped by '```'.
Output format:
---
<reasoning>
---
```kotlin
<code>
```

Now, translate this Java function to Kotlin, following the instructions above.
```java
{java_src_code}
```

Kotlin function signature:
{kotlin_signature}
"""



J2K_FIXUP_TEMPLATE = """You are an idiomatic code fixer with expert-level knowledge of Kotlin and Java. Your task is to fix a Kotlin function that has been partially-translated from Java.

You should aim to generate idiomatic Kotlin, while preserving the semantics of the original code as much as possible. Try to maintain type safety. Replace unavailable APIs with the Kotlin standard library or the closest equivalent where possible.

Explain your reasoning concisely. You should consider types, inputs, outputs, control flow (conditions and loops), libraries and APIs.

Example 1:
Code:
```kotlin
fun collectSquares(nums: IntArray): List<Int> {{
    val result: List<Int> = ArrayList()
    for (n in nums) {{
        (result as ArrayList).add(n * n)
    }}
    return result
}}
```
Reasoning:
---
The "collectSquares" function takes an array of integers "nums" and builds a list of their squares. Output type is List<Int>.
---

Example 2:
Code:
```kotlin
fun countZeros(numbers: List<Int>): Int {{
    var count = 0
    for (i in 0 until numbers.size()) {{
        if (numbers.get(i) == 0) {{
            count++
        }}
    }}
    return count
}}
```
Reasoning:
---
The function "countZeros" iterates through a list of integers and counts how many 0s appear. Input is List<Int>, output is an Int.
---

Example 3:
Code:
```kotlin
fun sumValues(values: List<Integer?>): Integer? {{
    var sum: Integer? = 0
    for (v in values) {{
        if (v != null) {{
            sum = sum!! + v
        }}
    }}
    return sum
}}
```
Reasoning:
---
The "sumValues" function sums a list of numbers. It takes a list named "values" as input and returns an integer as output.
---

Use reasoning like this to assist in fixing the code.

Output the reasoning in a block wrapped by '---'. Output the requested function and relevant imports in a fenced code block wrapped by '```'.
Output format:
---
<reasoning>
---
```kotlin
<code>
```

Now, fix this code where necessary, following the instructions above.
```kotlin
{kotlin_src_code}
```

Use this function signature:
{kotlin_signature}
"""